# 🎯 Notebook 2: SLI, SLO, SLA — and the Error Budget

Three letters that confuse everyone:

- **SLI** — Service Level **Indicator**. The number you measure. *e.g. "% of requests that returned 200 in under 300 ms"*.
- **SLO** — Service Level **Objective**. The target you set internally. *e.g. "99.9% over a rolling 28 days"*.
- **SLA** — Service Level **Agreement**. The contractual promise to a customer (with money or service credits if you miss it). Usually looser than the SLO so you have headroom.

Out of this falls the **error budget**: if your SLO is 99.9%, you are *allowed* to be bad **0.1%** of the time. That 0.1% is a budget you can spend on risky deploys and experiments.

## Learning objectives
- Tell **vanity metrics** from **user-visible SLIs**.
- Compute an SLI from raw request data and check it against an SLO.
- Track an error budget over time, and learn what **fast burn vs slow burn** look like.
- Pick metrics with the **RED method** (and know about USE for resources).

## 🪞 Vanity metrics vs user-visible SLIs

Beginner trap: alerting on metrics that *look* important but don't reflect what users feel.

| ❌ Vanity (machine-centric) | ✅ User-visible SLI |
|---|---|
| `cpu_percent` is high | `% of checkout requests succeeding < 300 ms` |
| `disk_io` is busy | `% of search results returned in < 1 s` |
| `gc_pause_ms` | `% of video starts within 2 s` |

CPU at 90% might be fine if every user is happy. CPU at 30% is **not** fine if half your users are getting 504s.

**Rule of thumb:** an SLI should be something you can put in a sentence a non-engineer understands.

## 🚦 The RED method (and a word on USE)

When you're staring at a blank dashboard for a new service, what do you measure first?

**RED** — for *services that handle requests* (APIs, web apps):
- **R**ate — requests per second
- **E**rrors — failed requests per second (or %)
- **D**uration — how long requests take (p50/p95/p99)

**USE** — for *resources* (CPU, disk, network, DB pool):
- **U**tilization — % time the resource is busy
- **S**aturation — work queued up waiting for the resource
- **E**rrors — error events from the resource

These are the **Four Golden Signals** in slightly different clothes (Google SRE adds *Saturation* to RED). Start every dashboard with these — *then* add domain-specific things.

## 🧪 Computing an SLI from raw data

Let's simulate one minute of traffic — 1000 requests — and compute the SLI: *"% of requests that succeeded in under 300 ms."*

In [ ]:
import random
random.seed(42)

requests = []
for _ in range(1000):
    latency = max(0, random.gauss(150, 80))
    success = (latency < 500) and (random.random() > 0.002)  # ~99.8% success
    requests.append({"latency_ms": latency, "ok": success})

good = sum(1 for r in requests if r["ok"] and r["latency_ms"] < 300)
total = len(requests)
sli = good / total

print(f"SLI (% requests OK and < 300 ms): {sli:.3%}")
print(f"SLO target: 99.0%  ->  {'✅ on track' if sli >= 0.99 else '❌ missed'}")

## 💰 The error budget

If your SLO is 99.0% over 30 days and you serve 1,000,000 requests in that window, your budget is:

```
budget = total × (1 − SLO) = 1_000_000 × 0.01 = 10_000 bad requests
```

Many teams use this rule:
> **If the budget is being burnt too fast, freeze risky deploys and focus on reliability. If the budget is unused, take more risks (chaos tests, faster releases).**

In [ ]:
TOTAL_REQUESTS = 1_000_000
SLO = 0.99
budget = TOTAL_REQUESTS * (1 - SLO)
print(f"30-day error budget: {int(budget):,} bad requests\n")

# Simulate 30 days, with one really bad day in the middle.
spent = 0
for day in range(1, 31):
    bad_today = 200 if day != 15 else 6000  # a big outage on day 15
    spent += bad_today
    pct = spent / budget
    bar = "█" * min(40, int(pct * 40))
    flag = "🚨" if pct > 1 else ("⚠️ " if pct > 0.7 else "  ")
    print(f"day {day:2d}: spent {spent:6d}/{int(budget)}  {bar:40s} {flag}")

## 🔥 Burn rate — fast burn vs slow burn

The **burn rate** is *how fast you're spending the budget* relative to "evenly over 30 days".

- Burn rate = **1.0** → you'll exactly use up the budget by day 30 (on plan).
- Burn rate = **10** → at this pace you'll exhaust the budget in 3 days. Page someone now.
- Burn rate = **0.5** → you have headroom; ship that risky feature.

Real SRE teams alert on **two** windows at once so they catch both kinds of trouble:

| Kind | What it looks like | Example alert |
|---|---|---|
| **Fast burn** | Big outage right now | `burn_rate(5m) > 14× and burn_rate(1h) > 14×` |
| **Slow burn** | Small constant regression | `burn_rate(6h) > 1× and burn_rate(3d) > 1×` |

Below we simulate both and compute a simple burn rate.

In [ ]:
# Helper: burn rate = (errors in window / requests in window) / (1 - SLO)
def burn_rate(errors, requests, slo):
    if requests == 0:
        return 0.0
    error_rate = errors / requests
    allowed_error_rate = 1 - slo
    return error_rate / allowed_error_rate

SLO = 0.99
RPS = 100  # requests per second

# Scenario A: fast burn — a big outage in the last 5 minutes (5% errors).
fast_window_s = 5 * 60
fast_reqs = RPS * fast_window_s
fast_errs = int(fast_reqs * 0.05)
print(f"FAST BURN: {fast_errs} errors in {fast_reqs} reqs over 5min")
print(f"  burn rate = {burn_rate(fast_errs, fast_reqs, SLO):.1f}×  -> 🚨 page on-call\n")

# Scenario B: slow burn — a tiny regression over 6 hours (1.2% errors).
slow_window_s = 6 * 3600
slow_reqs = RPS * slow_window_s
slow_errs = int(slow_reqs * 0.012)
print(f"SLOW BURN: {slow_errs} errors in {slow_reqs} reqs over 6h")
print(f"  burn rate = {burn_rate(slow_errs, slow_reqs, SLO):.1f}×  -> ⚠️  open a ticket\n")

# Scenario C: healthy — within budget.
healthy_errs = int(slow_reqs * 0.005)
print(f"HEALTHY:   {healthy_errs} errors in {slow_reqs} reqs over 6h")
print(f"  burn rate = {burn_rate(healthy_errs, slow_reqs, SLO):.1f}×  -> ✅ ship it")

**Why two windows?** A single short window catches loud outages but flaps on noise. A single long window catches slow regressions but reacts too late to a fire. Combining them gives you alerts that are both *fast* and *quiet*.

## 🤝 SLA vs SLO — why the gap?

People often ask, *"why are these different?"*

- **SLO** is what you aim for internally (e.g. 99.95%). You want headroom.
- **SLA** is what you promise customers in a contract (e.g. 99.9%). Missing it costs you money — service credits, refunds, reputational damage.

**Always set your SLO tighter than your SLA**, so you start sweating *before* you owe customers money.

## ✅ Recap

- **Pick SLIs that reflect what users feel**, not what your machines feel.
- Use the **RED** method for services and **USE** for resources to start any dashboard.
- The error budget turns reliability from a vibe into **math** — and the **burn rate** tells you *how urgent* a problem is.
- Alert on **two windows** (fast + slow) for fewer false alarms.

In the next notebook we build the actual collector that produces the numbers behind all of this.